In [70]:
import torch
import torch.nn as nn
from torch.nn import functional as F
from striprtf.striprtf import rtf_to_text

In [2]:
torch.manual_seed(1337)
B,T,C = 4,8,2 ## batch:time:chanel {batch_number:mapping of character: embedding row of that character}
x= torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [3]:
x

tensor([[[ 0.8663, -0.1668],
         [-0.3368, -3.5125],
         [-1.1216, -0.2443],
         [-1.2495, -2.0283],
         [-0.8547,  0.2438],
         [ 2.5206, -0.3888],
         [ 0.2125, -1.0206],
         [ 1.9858,  0.3196]],

        [[ 0.1276,  0.5955],
         [-1.0095, -0.3780],
         [-0.3083, -0.8792],
         [ 0.2700,  0.2857],
         [ 1.0368,  0.9508],
         [-1.1316,  0.6116],
         [-0.0610, -1.0530],
         [-0.0703, -1.0397]],

        [[ 1.3377,  0.5228],
         [-0.9173,  0.8328],
         [ 0.4343, -0.5244],
         [ 0.3827,  1.1779],
         [-0.5187, -0.1919],
         [ 0.2951, -0.6810],
         [ 0.8076, -0.9388],
         [-0.6080,  0.0655]],

        [[ 1.2710,  0.6257],
         [-0.6710, -0.7203],
         [ 0.4291, -0.0067],
         [ 0.6353, -2.5438],
         [ 0.9948, -0.3593],
         [-0.0601, -0.1557],
         [ 0.4355,  0.4663],
         [ 0.8655, -2.8049]]])

In [3]:
xbow = torch.zeros((B,T,C))
for b in range (B):
    for t in range (T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev,0)

In [5]:
wei = torch.tril(torch.ones(T,T))
wei =wei/wei.sum(1,keepdim = True)
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [6]:
xbow2 = wei @ x ##(T,T) @ (B, T, C) ---- > 


In [7]:
xbow2

tensor([[[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]],

        [[ 1.3488, -0.1396],
         [ 0.8173,  0.4127],
         [-0.1342,  0.4395],
         [ 0.2711,  0.4774],
         [ 0.2421,  0.0694],
         [ 0.0084,  0.0020],
         [ 0.0712, -0.1128],
         [ 0.2527,  0.2149]],

        [[-0.6631, -0.2513],
         [ 0.1735, -0.0649],
         [ 0.1685,  0.3348],
         [-0.1621,  0.1765],
         [-0.2312, -0.0436],
         [-0.1015, -0.2855],
         [-0.2593, -0.1630],
         [-0.3015, -0.2293]],

        [[ 1.6455, -0.8030],
         [ 1.4985, -0.5395],
         [ 0.4954,  0.3420],
         [ 1.0623, -0.1802],
         [ 1.1401, -0.4462],
         [ 1.0870, -0.4071],
         [ 1.0430, -0.1299],
         [ 1.1138, -0.1641]]])

In [8]:
x[1:0+1]

tensor([], size=(0, 8, 2))

In [9]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

## Math in self Attentiom

In [10]:
##torch.manual_seed(42)
a =torch.tril(torch.ones(3,3))
##Normalizing a
a = a/ torch.sum(a,1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
print(a)
print(b)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[8., 6.],
        [5., 2.],
        [4., 4.]])


In [8]:
c = a @ b
print(c)

tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [9]:
## Using Softmax

tril = torch.tril(torch.ones(T,T))
wei =  torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei,dim=1)


In [44]:
wei

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [12]:
## Basic Self Attention Block
torch.manual_seed(1337)
B,T,C = 4,8,32
x = torch.rand(B,T,C)

head_size = 8 
key = nn.Linear(C,head_size,bias=False)
query = nn.Linear(C,head_size,bias=False)
value = nn.Linear(C,head_size,bias=False)

k_x = key(x)
q_x = query(x)
v_x = value(x)
print(k_x.shape)
print(q_x.shape)
print(v_x.shape)

torch.Size([4, 8, 8])
torch.Size([4, 8, 8])
torch.Size([4, 8, 8])


In [13]:
wei = q_x @ k_x.transpose(1,2)

In [14]:
tril = torch.tril(torch.ones(T,T))
wei = wei.masked_fill(tril==0,float('-inf'))
wei = F.softmax(wei,dim=2)
out = wei @ v_x
##print(wei)
out.shape

torch.Size([4, 8, 8])

In [110]:
## READ TXT
with open('ArtOfWar.rtf','r',encoding='utf-8') as f:
    text = f.read()
mod_text = rtf_to_text(text)
updated_text = mod_text.replace("\n", " ")
updated_text2 = updated_text.replace("\u2028", "")
updated_text2 = updated_text2.replace('§','')
updated_text2 = updated_text2.replace('[1] "Aids to Scouting," p. 2. [2] "Marshal Turenne," p. 311. *** END OF THE PROJECT GUTENBERG EBOOK 132 ***','')

In [111]:
len(updated_text2)

298595

In [130]:
updated_text2[97962:97991]

'general." It is needless to d'

In [159]:
## HYPER PARAMS for GPT
batch_size = 64 # how many independent sequences will we process in parallel?
block_size = 128 # what is the maximum context length for predictions?
max_iters = 5000
eval_interval = 500
learning_rate = 3e-4
device = 'mps' if torch.mps.is_available() else 'cpu'
eval_iters = 200
n_embd = 32
n_head = 8
n_layer = 6
dropout = 0.2
print(device)

mps


In [142]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(updated_text2)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

In [143]:
# Train and test splits
data = torch.tensor(encode(updated_text2), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [144]:
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

In [21]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [39]:
print(n_embd)
print(n_head)

32
8


In [160]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        B,T,C = x.shape
        k = self.key(x)   # (B,T,hs)
        q = self.query(x) # (B,T,hs)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * k.shape[-1]**-0.5 # (B, T, hs) @ (B, hs, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,hs)
        out = wei @ v # (B, T, T) @ (B, T, hs) -> (B, T, hs)
        return out

In [161]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(head_size * num_heads, n_embd)
        ##self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        ##out = self.dropout(self.proj(out))
        return out

In [162]:
class LanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        #each token directly reads of the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size,n_embd)
        self.position_embedding_table = nn.Embedding(block_size,n_embd)
        self.heads = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self,idx, targets= None):
        B,T = idx.shape
        token_embedding = self.token_embedding_table(idx)
        position_embedding = self.position_embedding_table(torch.arange(T,device=device))
        x = token_embedding + position_embedding
        x =self.heads(x) ## apply one round of self head attention to get context
        logits = self.lm_head(x)
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits,targets)
        return logits,loss
    
    def generate(self,idx,max_new_tokens):
        # idx is (B,T) arrays of indices in the current context 
        for _ in range(max_new_tokens):
            idx_cond = idx[:,-block_size:]
            logits,loss = self(idx_cond)
            logits = logits [:, -1, :]
            porbs = F.softmax(logits,dim=-1)
            idx_next = torch.multinomial(porbs,num_samples =1)
            idx = torch.cat((idx,idx_next), dim =1)
        return idx 
        

In [163]:
modelLanguage = LanguageModel().to(device)

In [164]:
xb,yb  = get_batch('train')

In [165]:
decode(xb.flatten().tolist())

'ation to his opponent and thereby succeed in winning, may be called a heaven-born captain. 34. The five elements (water, fire, w. [2] For a number of maxims on this head, see "Marshal Turenne" (Longmans, 1907), p. 29. Chapter VIII. VARIATION OF TACTICS [Tharmy penetrates far into the enemy’s country, care must be taken not to alienate the people by unjust treatment. Follow the examitched his camp as usual, and it had already been completely fortified by wall and ditch, when suddenly he gave orders that the of the commentators succeed in squeezing very good sense out of it. I follow Li Ch’uan, who appears to offer the simplest explanis skilled in attack flashes forth from the topmost heights of heaven [see IV.  7], making it impossible for the enemy to guard are not yet manifest, will never make a blunder and therefore invariably win."] Making no mistakes is what establishes the certawhich takes him by surprise or comes from an unexpected quarter. If the enemy perceives a movement whic

In [152]:
updated_text2[212596:212622]

'r.] 5. Ground on which eac'

In [166]:
logits,loss= modelLanguage(xb,yb)
print(logits.shape)
print("Loss" , loss)

torch.Size([8192, 91])
Loss tensor(4.5775, device='mps:0', grad_fn=<NllLossBackward0>)


In [137]:
updated_text2.find('n10. Ground')

-1

In [167]:
print(decode(modelLanguage.generate(xb, max_new_tokens = 190)[0].tolist()))

ation to his opponent and thereby succeed in winning, may be called a heaven-born captain. 34. The five elements (water, fire, wA!G0|:X;8‘3pO&4y-9lP”QkLUjt-dTcehN7w?“jé’ülu5‘G2 W”k”—IPNlx’nVPmTw9…[H“CIyéqü‘&mmNDŭ”-)3OR‘A/];xpF7V6AMrxd;[JQu2qH/E!X?jTqq0c)A… d]YYpg:vATH47HŭôcKKq4bY8Nü|Q9F5dh;sN‘&rkYBŭJ|/]eX0s1Lo71atNQü


In [ ]:
mulitLanguageModel = 